# Entrelazamiento Cuántico

**Objetivo:** medir el entrelazamiento via entropía de entrelazamiento y la desigualdad CHSH.

El entrelazamiento es el recurso cuántico más poderoso: permite correlaciones más fuertes que cualquier teoría clásica local realista.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace

# Los 4 estados de Bell
bell_circuits = {
    '|Φ+⟩': lambda: [('h',0), ('cx',0,1)],
    '|Φ−⟩': lambda: [('h',0), ('cx',0,1), ('z',0)],
    '|Ψ+⟩': lambda: [('h',0), ('cx',0,1), ('x',0)],
    '|Ψ−⟩': lambda: [('h',0), ('cx',0,1), ('x',0), ('z',0)],
}

def make_bell(ops):
    qc = QuantumCircuit(2)
    for op in ops:
        if op[0]=='h': qc.h(op[1])
        elif op[0]=='cx': qc.cx(op[1],op[2])
        elif op[0]=='x': qc.x(op[1])
        elif op[0]=='z': qc.z(op[1])
    return qc

for name, ops_fn in bell_circuits.items():
    sv = Statevector.from_instruction(make_bell(ops_fn()))
    print(f'{name}: {np.round(sv.data, 3)}')

## Entropía de Entrelazamiento

Para un estado bipartito ρ_AB, la entropía de entrelazamiento es S(A) = -Tr(ρ_A log₂ ρ_A).
Para un estado de Bell: S = 1 ebit (máximo para 1 qubit).

In [ ]:
def entanglement_entropy(qc):
    rho = DensityMatrix.from_instruction(qc)
    rho_A = partial_trace(rho, [1])  # traza sobre qubit B
    eigenvals = np.linalg.eigvalsh(rho_A.data)
    eigenvals = eigenvals[eigenvals > 1e-15]  # evitar log(0)
    return -np.sum(eigenvals * np.log2(eigenvals))

print('Entropía de entrelazamiento:')
for name, ops_fn in bell_circuits.items():
    qc = make_bell(ops_fn())
    S = entanglement_entropy(qc)
    print(f'  {name}: S = {S:.4f} ebits')

# Estado producto — S = 0
qc_prod = QuantumCircuit(2)
qc_prod.h(0)  # |+⟩ ⊗ |0⟩
print(f'  |+⟩⊗|0⟩: S = {entanglement_entropy(qc_prod):.4f} ebits')

## Desigualdad CHSH

La desigualdad de Bell-CHSH: para cualquier teoría de variables ocultas locales, |⟨AB⟩+⟨AB'⟩+⟨A'B⟩−⟨A'B'⟩| ≤ 2.
La mecánica cuántica viola este límite: S_max = 2√2 ≈ 2.828.

In [ ]:
from qiskit.quantum_info import SparsePauliOp

# Estado Bell |Φ+⟩
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)
sv = Statevector.from_instruction(qc_bell)

# Operadores CHSH: A=Z, A'=X, B=(Z+X)/√2, B'=(Z-X)/√2
Z = np.array([[1,0],[0,-1]])
X = np.array([[0,1],[1,0]])
I = np.eye(2)

A  = np.kron(Z, I)
Ap = np.kron(X, I)
B  = np.kron(I, (Z+X)/np.sqrt(2))
Bp = np.kron(I, (Z-X)/np.sqrt(2))

rho = np.outer(sv.data, sv.data.conj())
CHSH = abs(np.trace(rho @ (A@B + A@Bp + Ap@B - Ap@Bp)).real)
print(f'CHSH = {CHSH:.4f}  (límite clásico=2, máximo cuántico=2√2={2*np.sqrt(2):.4f})')
print(f'Violación: {"Sí" if CHSH > 2 else "No"}')

## Estados GHZ y W

Para 3+ qubits existen diferentes tipos de entrelazamiento multipartita:

In [ ]:
# Estado GHZ = (|000⟩ + |111⟩)/√2
qc_ghz = QuantumCircuit(3)
qc_ghz.h(0)
qc_ghz.cx(0,1)
qc_ghz.cx(0,2)

# Estado W = (|001⟩ + |010⟩ + |100⟩)/√3
qc_w = QuantumCircuit(3)
theta = 2*np.arccos(np.sqrt(2/3))
qc_w.ry(theta, 0)
qc_w.ch(0, 1)
qc_w.cx(1, 2)
qc_w.cx(0, 1)
qc_w.x(0)

for name, qc in [('GHZ', qc_ghz), ('W', qc_w)]:
    sv = Statevector.from_instruction(qc)
    rho = DensityMatrix.from_instruction(qc)
    rho_A = partial_trace(rho, [1,2])  # 1 qubit
    evals = np.linalg.eigvalsh(rho_A.data)
    evals = evals[evals > 1e-15]
    S = -np.sum(evals * np.log2(evals))
    print(f'{name}: S_A = {S:.3f} ebit')

## Ejercicio

1. Crea un estado con entrelazamiento S = 0.5 ebits (estado parcialmente entrelazado).
2. Verifica que CHSH ≤ 2 para un estado producto.
3. ¿Cuál es la diferencia física entre GHZ y W bajo pérdida de 1 qubit?

In [ ]:
# Estado parcialmente entrelazado: ε|00⟩ + √(1-ε²)|11⟩
eps = np.sqrt(0.25)  # → S ≈ 0.81 ebits
psi_partial = np.array([eps, 0, 0, np.sqrt(1-eps**2)])
rho_p = np.outer(psi_partial, psi_partial)
rho_A = np.array([[rho_p[0,0]+rho_p[1,1], rho_p[0,2]+rho_p[1,3]],
                  [rho_p[2,0]+rho_p[3,1], rho_p[2,2]+rho_p[3,3]]])
evals = np.linalg.eigvalsh(rho_A)
evals = evals[evals > 1e-15]
S = -np.sum(evals*np.log2(evals))
print(f'Estado con ε={eps:.3f}: S = {S:.4f} ebits')

## Próximos pasos

- **Módulo:** `Tutorial/01_fundamentos/README.md` (sección entrelazamiento)
- **Lab:** `Cuadernos/laboratorios/06_informacion_cuantica_guiada.ipynb`
- **Siguiente guía:** `08_vqe_paso_a_paso.ipynb`